In [1]:
import numpy as np 
import pandas as pd
import tensorflow as tf

2024-03-31 14:51:49.049859: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-03-31 14:51:49.049959: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-03-31 14:51:49.180178: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
import os
import cv2
import imghdr
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [3]:
training_images_csv = pd.read_csv("/kaggle/input/bttai-nybg-2024/BTTAIxNYBG-train.csv")

In [4]:
training_images_filenames = training_images_csv['imageFile'].tolist()
training_image_labels = training_images_csv['classLabel'].tolist()

In [5]:
training_images = []

In [6]:
#for filename in training_images_filenames:
 #   img = cv2.imread("/kaggle/input/bttai-nybg-2024/BTTAIxNYBG-train/BTTAIxNYBG-train/" + filename)
  #  img = cv2.resize(img, (256, 256))
   # img = img / 255.0  
    #training_images.append(img)

In [7]:

# Specify the directory containing your images
training_images_dir = "/kaggle/input/bttai-nybg-2024/BTTAIxNYBG-train/BTTAIxNYBG-train"

# Define ImageDataGenerator for preprocessing and data augmentation
datagen = ImageDataGenerator(
    rescale=1./255          
)

# Specify batch size and target image size
batch_size = 32
target_size = (256, 256)  

# Define the generator for training data
train_generator = datagen.flow_from_dataframe(
    dataframe=training_images_csv,
    directory=training_images_dir,
    x_col="imageFile",   
    y_col="classLabel",      
    target_size=target_size,  
    batch_size=batch_size,
    class_mode='categorical'
)


Found 81946 validated image filenames belonging to 10 classes.


In [8]:
batch_data = next(train_generator)

# Unpack the batch data into images and labels
batch_images, batch_labels = batch_data

# Print the shapes of the input images and labels
print("Shape of input images batch:", batch_images.shape)
print("Shape of input labels batch:", batch_labels.shape)

Shape of input images batch: (32, 256, 256, 3)
Shape of input labels batch: (32, 10)


In [9]:
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout

In [10]:
model = Sequential()

In [11]:
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(256,256, 3)))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))


model.add(Flatten())


model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(10, activation='softmax'))

/opt/conda/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:99: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(


In [12]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [13]:
val_images_csv = pd.read_csv("/kaggle/input/bttai-nybg-2024/BTTAIxNYBG-validation.csv")
val_images_dir = "/kaggle/input/bttai-nybg-2024/BTTAIxNYBG-validation/BTTAIxNYBG-validation"

In [14]:
val_generator = datagen.flow_from_dataframe(
    dataframe=val_images_csv,
    directory=val_images_dir,
    x_col="imageFile",   
    y_col="classLabel",      
    target_size=target_size,  
    batch_size=batch_size,
    class_mode='categorical'
)

Found 10244 validated image filenames belonging to 10 classes.


In [15]:
model.fit(train_generator, epochs=12, validation_data=val_generator)

Epoch 1/12


/opt/conda/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
2024-03-31 14:55:17.286134: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 0: 5.22285, expected 4.50617
2024-03-31 14:55:17.286205: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 19: 4.83523, expected 4.11855
2024-03-31 14:55:17.286222: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 20: 6.14347, expected 5.42679
2024-03-31 14:55:17.286231: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 21: 5.72417, expected 5.00749
2024-03-31 14:55:17.286239: E external/local_xla/xla/service/gpu/buffer_comparator.cc

   1/2561 ━━━━━━━━━━━━━━━━━━━━ 10:41:13 15s/step - accuracy: 0.0938 - loss: 2.3061

I0000 00:00:1711896924.740511      70 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2115/2561 ━━━━━━━━━━━━━━━━━━━━ 3:31 473ms/step - accuracy: 0.6487 - loss: 1.1301

...

2560/2561 ━━━━━━━━━━━━━━━━━━━━ 0s 478ms/step - accuracy: 0.6633 - loss: 1.1040

2024-03-31 15:18:21.294600: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 129063: 5.35229, expected 4.67255
2024-03-31 15:18:21.294656: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 129317: 4.6381, expected 3.95836
2024-03-31 15:18:21.294666: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 129442: 5.36458, expected 4.68484
2024-03-31 15:18:21.294674: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 129443: 5.7503, expected 5.07056
2024-03-31 15:18:21.294682: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 129464: 5.45647, expected 4.77673
2024-03-31 15:18:21.294690: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 129486: 5.67042, expected 4.99069
2024-03-31 15:18:21.294698: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 129487: 5.79308, expected 5.11335
2024-03-31 15:18:21.29

2561/2561 ━━━━━━━━━━━━━━━━━━━━ 1393s 538ms/step - accuracy: 0.6633 - loss: 1.1039 - val_accuracy: 0.6829 - val_loss: 27.7451
Epoch 2/12


W0000 00:00:1711898302.299920      72 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


2561/2561 ━━━━━━━━━━━━━━━━━━━━ 907s 353ms/step - accuracy: 0.7694 - loss: 2.1759 - val_accuracy: 0.8787 - val_loss: 13.1770
Epoch 3/12
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 940s 366ms/step - accuracy: 0.8677 - loss: 1.1750 - val_accuracy: 0.8952 - val_loss: 11.3051
Epoch 4/12
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 1011s 394ms/step - accuracy: 0.9015 - loss: 0.4758 - val_accuracy: 0.8926 - val_loss: 22.9809
Epoch 5/12
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 1040s 405ms/step - accuracy: 0.9091 - loss: 4.1853 - val_accuracy: 0.9099 - val_loss: 2.3259
Epoch 6/12
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 1007s 392ms/step - accuracy: 0.9206 - loss: 1.6377 - val_accuracy: 0.9139 - val_loss: 4.4702
Epoch 7/12
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 927s 361ms/step - accuracy: 0.9291 - loss: 1.8570 - val_accuracy: 0.9179 - val_loss: 10.5327
Epoch 8/12
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 1035s 403ms/step - accuracy: 0.9021 - loss: 0.7229 - val_accuracy: 0.1007 - val_loss: 13.1005
Epoch 9/12
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 1055s 411ms/step - acc

In [16]:
test_images_csv = pd.read_csv("/kaggle/input/bttai-nybg-2024/BTTAIxNYBG-test.csv")
test_images_dir = "/kaggle/input/bttai-nybg-2024/BTTAIxNYBG-test/BTTAIxNYBG-test"

In [17]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = datagen.flow_from_dataframe(
    dataframe=test_images_csv,
    directory=test_images_dir,
    x_col="imageFile",   
    y_col= None,      
    target_size=target_size,  
    batch_size=batch_size,
    class_mode=None,
    shuffle = False
)

predictions = model.predict(test_generator)

predicted_labels = predictions.argmax(axis=1)


Found 30690 validated image filenames.


/opt/conda/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


959/960 ━━━━━━━━━━━━━━━━━━━━ 0s 386ms/step

2024-03-31 18:27:48.393561: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 0: 5.17171, expected 4.3613
2024-03-31 18:27:48.394660: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 1: 6.10962, expected 5.29921
2024-03-31 18:27:48.394685: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 2: 6.32575, expected 5.51534
2024-03-31 18:27:48.394698: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 3: 6.90323, expected 6.09282
2024-03-31 18:27:48.394709: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 6: 6.86856, expected 6.05815
2024-03-31 18:27:48.394720: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 7: 6.58906, expected 5.77865
2024-03-31 18:27:48.394730: E external/local_xla/xla/service/gpu/buffer_comparator.cc:1137] Difference at 8: 6.0516, expected 5.24119
2024-03-31 18:27:48.394740: E external/local_xla/xla/serv

960/960 ━━━━━━━━━━━━━━━━━━━━ 372s 386ms/step


W0000 00:00:1711909669.240949      71 graph_launch.cc:671] Fallback to op-by-op mode because memset node breaks graph update


In [18]:
predicted_labels

array([9, 9, 9, ..., 9, 9, 9])

In [19]:
uniqueID = test_images_csv['uniqueID']
classID = predicted_labels
predictions_df = pd.DataFrame({
    'uniqueID': uniqueID,
    'classID': classID
})
predictions_df

,uniqueID,classID
0,1,9
1,9,9
2,10,9
3,14,9
4,16,9
...,...,...
30685,122864,9
30686,122868,9
30687,122871,9
30688,122878,9


In [20]:
predictions_df.to_csv('predictions1.csv', index=False)